In [9]:
from collections import defaultdict
import pandas as pd
import os
import csv
from itertools import combinations
import kagglehub

dir_path = kagglehub.dataset_download("andrewmvd/spotify-playlists")
path = os.path.join(dir_path, "spotify_dataset.csv")

playlists = defaultdict(list)
transactions = []

# read the file in chunks
for chunk in pd.read_csv(path, chunksize=50000, on_bad_lines="skip"):

    # titles of csv columns were imported with extra ' 's
    # drop rows that have any null values in the row
    chunk.columns = [c.replace('"', '').strip() for c in chunk.columns]
    chunk = chunk.dropna(subset=["user_id", "playlistname", "trackname", "artistname"])

    # make the key unique with (userid, playlist) in case a playlist name is the same
    # make the value unique in case there's a song with the same title
    for row in chunk.itertuples(index=False):
        key = (row.user_id, row.playlistname)
        song = f"{row.trackname} - {row.artistname}"
        playlists[key].append(song)

# create a transactions array like the one we did in class
transactions = [frozenset(songs) for songs in playlists.values()]
print(len(transactions))

231560


In [10]:
# get a sense of how different support thresholds prune our items so we can pick the right one
from collections import Counter

# gives a count of how many transactions each song appears in aka support count
song_counts = Counter(item for t in transactions for item in t)

num_trans= len(transactions)  # 231560

# experimenting with some thresholds

print(f"Total number of items (non-duplicates) in across all transactions: {len(song_counts)}")

for min_support in [0.002, 0.001, 0.005, 0.01, 0.02]:
    passes_threshold = sum(1 for c in song_counts.values() if c / num_trans >= min_support)
    print(f"min_support={min_support:.4f} → {passes_threshold:,} items pass this threshold")


Total number of items (non-duplicates) in across all transactions: 2789644
min_support=0.0020 → 1,107 items pass this threshold
min_support=0.0010 → 3,747 items pass this threshold
min_support=0.0050 → 102 items pass this threshold
min_support=0.0100 → 3 items pass this threshold
min_support=0.0200 → 0 items pass this threshold


Looking at the output above we can see that choosing a support of 0.005, 0.010, or 0.020 prunes items in the transactions quite aggressively and doesn't leave many items to generate rules from. Additionally, we want to also avoid choosing a min_support that is too low which can cause Apriori to struggle with huge candidate sets. We see quite a large jump between 0.0005 and 0.001, so we chose to use 0.001 as our minimum threshold for this project.

Additionally, with min_support = 0.001 we see that only 3747 items pass the threshold which is a very small fraction out of the 2789644 items. Thus, we can deduce that there may be a lot of rare songs and have decided to clean and process the transactions arrar accordingly (also helps with Apriori algorithm execution a lot). To try and keep as many frequent itemsets as possible (since we already cut data down to help Apriori execute), we experimented with low minimum support values.



In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# filtering to pre-prune before Apriori (otherwise will take very long)

# remove playlists with only one song (can't generate rules from)
# grab the top 200 songs using support counts from song_counts from before
top_100 = {song for song, count in song_counts.most_common(100)}
print("got top songs!")

# keep only the top 200 in the filtered_transactions that we'll use for Apriori
filtered_transactions = [[s for s in t if s in top_100] for t in transactions]
print("filtered based off top songs")

filtered_transactions = [t for t in filtered_transactions if len(t) >= 2] # removes playlists with only one song 
print(len(filtered_transactions))

te = TransactionEncoder()
df = pd.DataFrame(te.fit_transform(filtered_transactions), columns=te.columns_)
freq_items = apriori(df, min_support=0.0005, use_colnames=True, verbose=1, max_len=3)
freq_items['length'] = freq_items['itemsets'].apply(len)

print(f"Frequent itemsets found: {len(freq_items):,}")

got top songs!
filtered based off top songs
23339
Processing 485100 combinations | Sampling itemset size 3
Frequent itemsets found: 139,985


In [12]:
rules = association_rules(freq_items, metric="confidence", min_threshold=0.70)
rules = rules.sort_values('lift', ascending=False)
print(f"Rules found: {len(rules):,}")
print(rules[['antecedents','consequents','support','confidence','lift']].head(10))

Rules found: 296
                                           antecedents  \
202  (Get Lucky - Daft Punk, Instant Crush - Daft P...   
4    (A Sky Full of Stars - Coldplay, Fix You - Col...   
8    (A Sky Full of Stars - Coldplay, Yellow - Cold...   
201  (Lose Yourself to Dance - Daft Punk, Get Lucky...   
74   (Lose Yourself to Dance - Daft Punk, Smells Li...   
269  (The Scientist - Coldplay, Summer - Calvin Har...   
23   (Smells Like Teen Spirit - Nirvana, Breezebloc...   
196            (Fix You - Coldplay, Yellow - Coldplay)   
295       (Yellow - Coldplay, Viva La Vida - Coldplay)   
294  (The Scientist - Coldplay, Viva La Vida - Cold...   

                              consequents   support  confidence       lift  
202  (Lose Yourself to Dance - Daft Punk)  0.028450    0.915862  16.857496  
4                      (Magic - Coldplay)  0.005184    0.724551  16.013535  
8                      (Magic - Coldplay)  0.004028    0.717557  15.858967  
201           (Instant Crush - Daft 

In [13]:
# we have our strong rules above. now we want to build how we recommend using that.
# first filter out metrics above a certain threshold
# for the input playlist, if it has the antencdent we recommmend the consequent if not already contained
def recommend_songs(input_playlist, rules, top_n=50):
    filtered_rules = rules[(rules['support'] >= 0.005) & (rules['lift'] >= 2.0)].copy()
    recommendations = []
    

    for _, row in filtered_rules.iterrows():
        antecedent = set(row['antecedents']) # songs we have
        consequent = set(row['consequents']) # songs we would recommend

        # if the playlist contains all the songs we have, recommend if not already in the playlist
        if antecedent.issubset(input_playlist):
            for song in consequent:
                if song not in input_playlist:
                    recommendations.append((song, antecedent, consequent, row['confidence'], row['lift']))

    # sort by lift (or confidence)
    recommendations.sort(key=lambda x: x[4], reverse=True)  # lift
    return recommendations[:top_n]

input_playlist = {
    "Yellow - Coldplay",
    "Fix You - Coldplay",
    "Viva La Vida - Coldplay"
}

recommended_songs = recommend_songs(input_playlist, rules)

print(f"=== Final Recommended Songs For an Example Playlist ===")
print(f"{'title':30} {'antecedent':50} {'consequent':29} {'confidence':20} {'lift'}")
for song, antecedent, consequent, conf, lift in recommended_songs:

    print(f"{song:30} {str(antecedent):30} {str(consequent):30} {conf:15.2f} {lift:15.2f}")



=== Final Recommended Songs For an Example Playlist ===
title                          antecedent                                         consequent                    confidence           lift
The Scientist - Coldplay       {'Fix You - Coldplay', 'Yellow - Coldplay'} {'The Scientist - Coldplay'}              0.77           14.79
The Scientist - Coldplay       {'Yellow - Coldplay', 'Viva La Vida - Coldplay'} {'The Scientist - Coldplay'}              0.76           14.70
The Scientist - Coldplay       {'Fix You - Coldplay', 'Viva La Vida - Coldplay'} {'The Scientist - Coldplay'}              0.72           13.91


In [14]:
# training and validation on playlists (hide 20%, run Apriori/recommend_songs() on the other 80%)
import random
def split_playlist(transactions, test_ratio=0.2):
    playlists = list(transactions) 
    random.shuffle(playlists) # randomize the split
    
    split = int(len(playlists) * (1 - test_ratio))

    # first 80% of all transactions are training, rest is testing.
    train = playlists[:split]
    test = playlists[split:]
    return train, test

training_set, test_set = split_playlist(transactions)
print(f"Training set length: {len(training_set):,}")
print(f"Test set length: {len(test_set):,}")

Training set length: 185,248
Test set length: 46,312


In [15]:
# run Apriori on the training set
# generating rules form the found frequent itemsets of songs

# count songs using training data only
training_song_counts = Counter(item for t in training_set for item in t)

# keep only top 100 songs from the training set
top_100_train = {song for song, count in training_song_counts.most_common(100)}
print("got top training songs!")

# filter training transactions
filtered_training = [[s for s in t if s in top_100_train] for t in training_set]
print("filtered training transactions")

# remove short playlists
filtered_training = [t for t in filtered_training if len(t) >= 2]

# encode
te = TransactionEncoder()
df = pd.DataFrame(te.fit_transform(filtered_training), columns=te.columns_)
training_freq_items = apriori(df, min_support=0.0001, use_colnames=True, verbose=1, max_len=3)
training_freq_items['length'] = training_freq_items['itemsets'].apply(len)

training_rules = association_rules(training_freq_items, metric='confidence', min_threshold=0.70)
print("Training transactions:", len(training_set))
print("Frequent itemsets found:", len(training_freq_items))
print("Rules found:", len(training_rules))

got top training songs!
filtered training transactions
Processing 485100 combinations | Sampling itemset size 3
Training transactions: 185248
Frequent itemsets found: 166386
Rules found: 372


In [ ]:
# out of all the recommended songs, how many are in the playlist
# returns a ratio for ONE playlist
def recommendation_score(recommended_songs, hidden_song_set, rec_num=50):
    top_recs = recommended_songs[:rec_num]
    if not top_recs:
        return 0
    
    hits = sum(1 for r in top_recs if r[0] in hidden_song_set)
    return hits/len(top_recs)

def evaluate_model(transactions, rules, top_songs, k=3):
    scores = []

    for i, playlist in enumerate(transactions):
        playlist = frozenset(s for s in playlist if s in top_songs) 

        if len(playlist) < 2:
            continue

        if i % 1000 == 0:
            print(f"Processed {i} playlists")

        # split the playlist and feed training set to model
        playlist = list(playlist)
        random.shuffle(playlist)
        split = int(len(playlist) * 0.8)
        input_playlist = playlist[:split]
        hidden_songs = set(playlist[split:])

        recs = recommend_songs(input_playlist, rules)
        score = recommendation_score(recs, hidden_songs)
        scores.append(score)

    return sum(scores) / len(scores) if scores else 0

# average hit score over all playlists
overall_score = evaluate_model(test_set, training_rules, top_100_train)
print("Overall Hits:", overall_score)

Processed 5000 playlists
Processed 10000 playlists
Processed 14000 playlists
Processed 22000 playlists
Processed 26000 playlists
Overall Hits: 0.07508965129882215
